# Inference / Checkpoint Evaluation (Colab / Local)

Loads a checkpoint trained by `03_train_colab.ipynb` and evaluates it on the test split
two ways: the standard **single-window** sample each clip got during training, and
**multi-clip** (`MultiClipWorkoutDataset` + `evaluate_multi_clip`, in `training_utils.py`) -
averaging predictions over `NUM_CLIPS` windows spanning the whole clip. Compares the two so
we can see whether multi-clip actually helps on this dataset before relying on it.

Does not retrain anything - only needs `artifacts/checkpoints/*.ckpt` to already exist
(from a previous run of `03_train_colab.ipynb`), locally or on Colab.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

# Works around a known Windows conda/pip OpenMP DLL conflict (harmless elsewhere).
os.environ.setdefault('KMP_DUPLICATE_LIB_OK', 'TRUE')

GIT_URL = 'https://github.com/hagairavid18/beilinson.git'
GIT_BRANCH = 'main'

try:
    import google.colab  # noqa: F401
    PROJECT_ROOT = Path('/content/beilinson')
    if PROJECT_ROOT.exists():
        subprocess.check_call(['git', '-C', str(PROJECT_ROOT), 'pull'])
    else:
        subprocess.check_call(['git', 'clone', '--branch', GIT_BRANCH, GIT_URL, str(PROJECT_ROOT)])
except ImportError:
    # Not on Colab (e.g. a local kernel) - use the repo checkout we're already in.
    PROJECT_ROOT = Path.cwd()

os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))
print('Project root:', PROJECT_ROOT)
print('Has data already:', any((PROJECT_ROOT / 'data').glob('*/*')))
print('Has checkpoints already:', any((PROJECT_ROOT / 'artifacts' / 'checkpoints').glob('*.ckpt')))

In [ ]:
import subprocess
import sys

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'])

In [ ]:
import json

import yaml
import pandas as pd
import lightning.pytorch as pl
from torch.utils.data import DataLoader

from dataset import MultiClipWorkoutDataset, WorkoutSequenceDataset
from model import SequenceClassifier
from pytorch_lightning import WorkoutLightningModule
from training_utils import ensure_artifacts, ensure_dataset, evaluate_multi_clip

## Config

In [ ]:
with open(PROJECT_ROOT / 'configs' / 'base.yaml', 'r', encoding='utf-8') as handle:
    CONFIG = yaml.safe_load(handle)

# How many windows to average per clip for the multi-clip evaluation below.
NUM_CLIPS = 5

CONFIG

## Dataset

In [ ]:
class_names = ensure_dataset(PROJECT_ROOT)
print(f'{len(class_names)} classes:', class_names)

## Pick a checkpoint

Defaults to the `best_model_path` recorded by the last `03_train_colab.ipynb` run
(`artifacts/training_summary.json`); falls back to the most recently modified `.ckpt` in
`artifacts/checkpoints/`. Set `CHECKPOINT_PATH` yourself to evaluate a different one.

In [ ]:
summary_path = PROJECT_ROOT / 'artifacts' / 'training_summary.json'
checkpoint_dir = PROJECT_ROOT / 'artifacts' / 'checkpoints'

CHECKPOINT_PATH = None
if summary_path.exists():
    with open(summary_path, 'r', encoding='utf-8') as handle:
        best_model_path = json.load(handle).get('best_model_path', '')
    if best_model_path and Path(best_model_path).exists():
        CHECKPOINT_PATH = Path(best_model_path)

if CHECKPOINT_PATH is None:
    checkpoints = sorted(checkpoint_dir.glob('*.ckpt'), key=lambda p: p.stat().st_mtime)
    if not checkpoints:
        raise FileNotFoundError(f'No checkpoints found under {checkpoint_dir} - run 03_train_colab.ipynb first.')
    CHECKPOINT_PATH = checkpoints[-1]

print('Using checkpoint:', CHECKPOINT_PATH)

## Load model from checkpoint

`WorkoutLightningModule` ignores `model` in `save_hyperparameters`, so the architecture has
to be rebuilt from `configs/base.yaml` (must match what the checkpoint was trained with) and
passed in explicitly - only `lr`/`weight_decay` are restored automatically from the checkpoint.

In [ ]:
artifacts = ensure_artifacts(CONFIG, PROJECT_ROOT)
label_map = pd.read_csv(artifacts['label_map'])
num_classes = int(label_map['label_id'].nunique())

model_cfg = CONFIG['model']
model = SequenceClassifier(
    num_classes=num_classes,
    in_channels=model_cfg['in_channels'],
    hidden_dims=tuple(model_cfg['hidden_dims']),
    embedding_dim=model_cfg['embedding_dim'],
    dropout=model_cfg['dropout'],
    temporal_pooling=model_cfg['temporal_pooling'],
)
lit_module = WorkoutLightningModule.load_from_checkpoint(str(CHECKPOINT_PATH), model=model)
lit_module.eval()

## Evaluate: single window (standard)

Same sampling every test clip got during training - one `sequence_len`-frame window,
evenly spaced across the whole clip.

In [ ]:
data_cfg = CONFIG['data']
test_dataset = WorkoutSequenceDataset(
    artifacts['sequence_manifest'], PROJECT_ROOT, split='test', image_size=data_cfg['image_size'],
)
test_loader = DataLoader(test_dataset, batch_size=data_cfg['batch_size'], shuffle=False)

trainer = pl.Trainer(logger=False, enable_checkpointing=False, enable_progress_bar=True)
single_window_results = trainer.test(lit_module, dataloaders=test_loader, verbose=True)

## Evaluate: multi-clip

`NUM_CLIPS` windows per test clip (set above), predictions averaged per clip.

In [ ]:
multi_clip_dataset = MultiClipWorkoutDataset(
    frame_manifest_path=artifacts['frame_manifest'],
    label_map_path=artifacts['label_map'],
    data_root=PROJECT_ROOT,
    split='test',
    sequence_len=data_cfg['sequence_len'],
    num_clips=NUM_CLIPS,
    image_size=data_cfg['image_size'],
)
multi_clip_accuracy, multi_clip_results = evaluate_multi_clip(
    lit_module, multi_clip_dataset, batch_size=max(1, data_cfg['batch_size'] // NUM_CLIPS),
)
print(f'Multi-clip (num_clips={NUM_CLIPS}) test accuracy: {multi_clip_accuracy:.4f}')

## Compare

In [ ]:
single_window_accuracy = single_window_results[0]['test_acc']
delta = multi_clip_accuracy - single_window_accuracy

pd.DataFrame(
    [
        {'strategy': 'single window', 'num_clips': 1, 'test_acc': single_window_accuracy},
        {'strategy': 'multi-clip', 'num_clips': NUM_CLIPS, 'test_acc': multi_clip_accuracy},
    ]
)